# 🎮 Python Arcade Project — Arcade_Ntbk
**Created by:** **Group 7:**
*1. Anjali Nupur Lakra (8)*, *2. Prithvish Mohanty(44)*, *3. Atman Nayak(16)*,  *4. Divya Aradhana Puntia(22)*

Run the cells below in order (or Run All). After the last cell (Arcade Menu) runs, you'll get a numbered menu to pick a game. Each game runs in the notebook using standard `input()` prompts.

---


In [1]:

# --- Hand Cricket ---
import random, time

def hand_cricket_main():
    print("🏏 Welcome to Hand Cricket!")
    print("Rules: Choose a number between 1 and 6. If it matches the computer, you're OUT!\n")

    wickets = 1
    overs = 1

    # Ask user for simple settings
    try:
        wickets = int(input("Enter number of wickets (e.g., 1-10): ").strip() or "1")
        overs = int(input("Enter number of overs (e.g., 1-10): ").strip() or "1")
    except Exception:
        print("Invalid input, using defaults 1 wicket, 1 over.")

    total_balls = overs * 6
    print(f"\nGame set: {wickets} Wickets, {overs} Overs ({total_balls} balls total).")

    # Toss (odd/even)
    print("\n--- Toss Time (Odd or Even) ---")
    while True:
        choice_input = input("Choose 'odd' or 'even': ").strip().lower()
        if choice_input in ['odd','o']:
            player_choice = 'odd'
            break
        if choice_input in ['even','e']:
            player_choice = 'even'
            break
        print("Invalid choice. Type 'odd' or 'even'.")

    player_num = None
    while player_num is None:
        try:
            player_num = int(input("Enter your number for toss (0-6): "))
            if not (0 <= player_num <= 6):
                print("Number must be 0-6")
                player_num = None
        except ValueError:
            print("Enter a number 0-6")

    comp_num = random.randint(0,6)
    print(f"You showed: {player_num} | Computer showed: {comp_num}")
    total_sum = player_num + comp_num
    toss_result = 'odd' if total_sum % 2 else 'even'
    print(f"The sum is {total_sum}, which is {toss_result.upper()}.")

    if toss_result == player_choice:
        print("You win the toss!")
        while True:
            ch = input("Choose to 'bat' or 'bowl': ").strip().lower()
            if ch in ['bat','b']:
                toss_winner, first_choice = "Player", "bat"
                break
            if ch in ['bowl','w']:
                toss_winner, first_choice = "Player", "bowl"
                break
            print("Invalid, type 'bat' or 'bowl'.")
    else:
        print("Computer wins the toss!")
        toss_winner = "Computer"
        first_choice = random.choice(['bat','bowl'])
        print(f"Computer chooses to {first_choice} first.")

    if first_choice in ['bat','b'] and first_choice == 'bat':
        batting_first = toss_winner
    else:
        batting_first = 'Computer' if toss_winner == 'Player' else 'Player'
    batting_second = 'Computer' if batting_first == 'Player' else 'Player'

    print(f"\n1st Innings: {batting_first} bats first")

    #run_innings
    def run_innings(batting_team, bowling_team, max_wickets, max_overs, target=None):
        current_runs = 0; wickets_lost = 0; balls_bowled = 0; max_balls = max_overs * 6
        while wickets_lost < max_wickets and balls_bowled < max_balls:
            print("-" * 30)
            print(f"Score: {current_runs}/{wickets_lost} | Ball: {balls_bowled+1} of {max_balls}")
            if target is not None:
                print(f"Needs: {max(0, target - current_runs)} to win")

            if batting_team == "Player":
                try:
                    batter_num = int(input("Enter your move (1-6): "))
                except ValueError:
                    print("Invalid, enter 1-6"); continue
                bowler_num = random.randint(1,6)
                print(f"You Batted: {batter_num} | Computer Bowled: {bowler_num}")
            else:
                try:
                    bowler_num = int(input("Enter your bowling move (1-6): "))
                except ValueError:
                    print("Invalid, enter 1-6"); continue
                batter_num = random.randint(1,6)
                print(f"Computer Batted: {batter_num} | You Bowled: {bowler_num}")

            if batter_num == bowler_num:
                print("OUT!")
                wickets_lost += 1
            else:
                current_runs += batter_num
                print(f"{batting_team} scores {batter_num} runs!")

            balls_bowled += 1

            if target is not None and current_runs >= target:
                print(f"{batting_team} reached the target!")
                break

        print(f"End innings: {current_runs}/{wickets_lost} in {balls_bowled//6}.{balls_bowled%6} overs")
        return current_runs, balls_bowled

    score1, _ = run_innings(batting_first, batting_second, wickets, overs)
    target = score1 + 1
    print(f"\nTarget for {batting_second}: {target}")

    score2, _ = run_innings(batting_second, batting_first, wickets, overs, target)
    # result
    if score2 >= target:
        print(f"{batting_second} wins!")
    elif score2 < score1:
        print(f"{batting_first} wins by {score1 - score2} runs")
    else:
        print("It's a tie!")

    input("\\nPress Enter to return to the Arcade Menu...")


In [2]:

# --- Hangman ---
import random, time
try:
    import pandas as pd
    _HAS_PANDAS = True
except Exception:
    _HAS_PANDAS = False

# Try NLTK words if available otherwise fallback list
try:
    import nltk
    from nltk.corpus import words as nltk_words
    word_list_source = [w.lower() for w in nltk_words.words() if 4 <= len(w) <= 10]
    if not word_list_source:
        raise LookupError
except Exception:
    word_list_source = [
        "apple", "banana", "cherry", "dragon", "forest", "garden", "hacker",
        "island", "jungle", "kettle", "library", "mountain", "nectar",
        "ocean", "puzzle", "rocket", "silver", "turkey", "umbrella", "violet"
    ]

WORDS = list({w.lower() for w in word_list_source if 4 <= len(w) <= 10})

def play_round(secret_word, max_lives=6):
    word = secret_word.lower()
    display = ["_"] * len(word)
    lives = max_lives
    guessed = set()
    record = []
    attempt_no = 0
    while lives > 0 and "_" in display:
        attempt_no += 1
        print("\\nWord:", " ".join(display))
        print(f"Lives left: {lives} | Guessed: {sorted(list(guessed))}")
        guess = input("Enter a letter: ").lower().strip()

        if not guess.isalpha() or len(guess) != 1:
            print("Please enter a single alphabet letter.")
            attempt_no -= 1
            continue
        if guess in guessed:
            print("You already guessed that!")
            attempt_no -= 1
            continue
        guessed.add(guess)
        if guess in word:
            print("✅ Good guess!")
            for i, ch in enumerate(word):
                if ch == guess:
                    display[i] = guess
        else:
            lives -= 1
            print("❌ Wrong guess!")
        record.append([attempt_no, guess, " ".join(display), lives])

    won = "_" not in display
    wrong_guesses = max_lives - lives
    return won, wrong_guesses, record, word

def hangman_main():
    print("🎯 Welcome to Hangman!")
    name = input("Enter your name: ").strip() or "Player"
    rounds = 3
    wins = 0
    overall = []
    for r in range(1, rounds + 1):
        print(f"\\n----- Round {r} -----")
        secret = random.choice(WORDS)
        print(f"(Hint: the word has {len(secret)} letters)")
        won, wrong_guesses, record, word = play_round(secret, max_lives=6)
        if won:
            print(f"\\n🎉 Congrats {name}! You guessed the word: {word}")
            result = "Won"
            wins += 1
        else:
            print(f"\\n💀 Sorry {name}, you lost! The word was: {word}")
            result = "Lost"
        print(f"\\nRound {r} Scoreboard:\\n")
        if _HAS_PANDAS and record:
            df = pd.DataFrame(record, columns=["Attempt","Guessed Letter","Word Progress","Lives Left"])
            print(df.to_string(index=False))
        else:
            if record:
                print(f"{'Attempt':>7} | {'Guess':>5} | {'Progress':>20} | {'Lives':>5}")
                print("-"*50)
                for row in record:
                    print(f"{row[0]:7} | {row[1]:5} | {row[2]:20} | {row[3]:5}")
            else:
                print("(No attempts were made.)")
        overall.append([r, word, result, wrong_guesses])
    print("\\n===== FINAL GAME SUMMARY =====\\n")
    if _HAS_PANDAS and overall:
        summary_df = pd.DataFrame(overall, columns=["Round","Word","Result","Wrong Guesses"])
        print(summary_df.to_string(index=False))
    else:
        print(f"{'Round':>5} | {'Word':>10} | {'Result':>6} | {'Wrong':>6}")
        print("-"*40)
        for row in overall:
            print(f"{row[0]:5} | {row[1]:10} | {row[2]:6} | {row[3]:6}")
    print("\\n===== FINAL RESULT =====")
    if wins * 2 > rounds:
        print("🏆", name, "wins the game with", wins, "out of", rounds, "rounds!")
    elif wins * 2 == rounds:
        print("🤝 It's a tie! You won", wins, "rounds.")
    else:
        print("💀", name, "lost the game. Better luck next time!")
    input("\\nPress Enter to return to the Arcade Menu...")


In [3]:

# --- Battleship ---
import random, time

def print_board(board):
    for row in board:
        print(" ".join(row))
    print()

def create_board(size):
    return [["~"] * size for _ in range(size)]

def random_ship_positions(size, num_ships):
    ships = set()
    while len(ships) < num_ships:
        ships.add((random.randint(0, size - 1), random.randint(0, size - 1)))
    return ships

def battleship_main():
    print("\n🚢 Welcome to Mini Battleship!")
    print("Try to sink all the computer's ships!\n")
    time.sleep(1)

    size = 5; num_ships = 3; turns = 8
    board = create_board(size)
    ships = random_ship_positions(size, num_ships)
    hits = set()
    print_board(board)

    for turn in range(1, turns + 1):
        print(f"Turn {turn} of {turns}")
        try:
            row = int(input(f"Enter row (0-{size-1}): "))
            col = int(input(f"Enter col (0-{size-1}): "))
        except ValueError:
            print("Invalid input. Enter numbers only!")
            continue
        if not (0 <= row < size and 0 <= col < size):
            print(f"Coordinates must be between 0 and {size-1}. Try again.")
            continue
        if (row, col) in hits:
            print("You already tried that spot!")
            continue
        if (row, col) in ships:
            print("💥 Hit!")
            board[row][col] = "X"
            hits.add((row, col))
            if hits == ships:
                print_board(board)
                print("\n🏆 You sunk all the ships! You win!")
                input("\\nPress Enter to return to the Arcade Menu...")
                return
        else:
            print("🌊 Miss!")
            board[row][col] = "O"
            hits.add((row, col))
        print_board(board)
        time.sleep(0.5)

    print("\n💀 Game Over!")
    print("Here were the ships:")
    for (r, c) in ships:
        board[r][c] = "S"
    print_board(board)
    input("\\nPress Enter to return to the Arcade Menu...")


In [4]:

# --- Blackjack ---
import random, time

def draw_card():
    cards = ['A','2','3','4','5','6','7','8','9','10','J','Q','K']
    card = random.choice(cards)
    if card in ['J','Q','K']:
        value = 10
    elif card == 'A':
        value = 11
    else:
        value = int(card)
    return card, value

def calculate_total(hand):
    total = sum(value for _, value in hand)
    for card, value in hand:
        if total > 21 and card == 'A':
            total -= 10
    return total

def display_hand(name, hand, hide_first=False):
    if hide_first:
        print(f"{name}'s Hand: [?, {hand[1][0]}]")
    else:
        cards = [card for card, _ in hand]
        total = calculate_total(hand)
        print(f"{name}'s Hand: {cards}  | Total: {total}")

def blackjack_main():
    print("\n🃏 Welcome to Mini Blackjack! 🃏")
    time.sleep(0.5)
    player_hand = [draw_card(), draw_card()]
    dealer_hand = [draw_card(), draw_card()]
    display_hand("Dealer", dealer_hand, hide_first=True)
    display_hand("Player", player_hand)

    while True:
        total = calculate_total(player_hand)
        if total > 21:
            print("💥 You busted! Dealer wins.")
            print("\nReturning to Arcade Menu...")
            time.sleep(1)
            return
        choice = input("Do you want to 'hit' or 'stand'? ").strip().lower()
        if choice in ['hit','h']:
            new_card = draw_card()
            player_hand.append(new_card)
            print(f"You drew: {new_card[0]}")
            display_hand("Player", player_hand)
        elif choice in ['stand','s']:
            break
        else:
            print("Invalid input. Type 'hit' or 'stand'.")

    print("\nDealer's Turn:")
    time.sleep(0.5)
    display_hand("Dealer", dealer_hand)

    while calculate_total(dealer_hand) < 17:
        new_card = draw_card()
        dealer_hand.append(new_card)
        print(f"Dealer draws: {new_card[0]}")
        time.sleep(0.5)
        display_hand("Dealer", dealer_hand)

    player_total = calculate_total(player_hand)
    dealer_total = calculate_total(dealer_hand)

    print("\n🎯 Final Results:")
    display_hand("Player", player_hand)
    display_hand("Dealer", dealer_hand)

    if dealer_total > 21:
        print("🏆 Dealer busted! You win!")
    elif player_total > dealer_total:
        print("🏆 You win!")
    elif player_total < dealer_total:
        print("💻 Dealer wins!")
    else:
        print("🤝 It's a tie!")

    print("\nReturning to Arcade Menu...")
    time.sleep(1)


In [5]:

# --- Typing Speed Game ---
import time, random
# Attempt to load brown corpus if available, else fallback sentences
try:
    import nltk
    from nltk.corpus import brown
    nltk.download('brown', quiet=True)
    sentences_corpus = [" ".join(s) for s in brown.sents() if 5 <= len(s) <= 15]
    if not sentences_corpus:
        raise Exception("Empty corpus")
except Exception:
    sentences_corpus = [
        "The quick brown fox jumps over the lazy dog.",
        "Python makes programming fun and easy.",
        "Typing fast requires focus and accuracy.",
        "Never stop learning new things every day.",
        "Code is like humor; it’s better when it works."
    ]

def typing_speed_main():
    print("\n⌨️ Welcome to the Typing Speed Game!")
    time.sleep(0.5)
    sentence = random.choice(sentences_corpus)
    print("Your sentence:\n")
    print(f"👉 {sentence}\n")
    input("Press Enter when you're ready to start typing... ")
    start_time = time.time()
    user_input = input("\nType here: ")
    end_time = time.time()
    total_time = round(end_time - start_time, 2)
    words = len(sentence.split())
    speed = round((words / total_time) * 60, 2) if total_time>0 else 0
    correct_chars = sum(1 for i, c in enumerate(user_input) if i < len(sentence) and c == sentence[i])
    accuracy = round((correct_chars / len(sentence)) * 100, 2)
    print("\n--- Results ---")
    print(f"Time Taken: {total_time} seconds")
    print(f"Speed: {speed} words per minute")
    print(f"Accuracy: {accuracy}%")
    input("\nPress Enter to return to the Arcade Menu...")


In [6]:

# --- Rock Paper Scissors ---
import random
def rps_main():
    points1 = 0; points2 = 0; rounds_played = 0; total_rounds = 3
    choices_emojis = {1: '🪨', 2: '✋', 3: '✂'}
    while rounds_played < total_rounds:
        print("\nEnter from the following choices:")
        try:
            choice1 = int(input("(1) ROCK (2) PAPER (3) SCISSORS\n"))
        except ValueError:
            print("Invalid input. Please enter 1, 2, or 3."); continue
        if choice1 not in choices_emojis:
            print("Invalid choice. Please enter 1, 2, or 3."); continue
        choice2 = random.randint(1,3)
        print(f"You: {choices_emojis[choice1]}  |  Computer: {choices_emojis[choice2]}")
        if choice1 == choice2:
            print("It's a tie!")
        elif (choice1==1 and choice2==3) or (choice1==2 and choice2==1) or (choice1==3 and choice2==2):
            points1 += 1; print("You win this round!")
        else:
            points2 += 1; print("Computer wins this round!")
        print(f"Score: You - {points1}, Computer - {points2}")
        rounds_played += 1
    print("\n--- Game Over ---")
    if points1 > points2: print(f"🏆 You won with {points1} points!")
    elif points2 > points1: print(f"💻 Computer won with {points2} points!")
    else: print(f"🤝 It's a draw! Both scored {points1} points!")
    input("\nPress Enter to return to the Arcade Menu...")


In [7]:

# --- Tic Tac Toe ---
import numpy as np, random
def tictactoe_main():
    def create_board(): return np.full((3,3),' ')
    def print_board(b):
        print("\n")
        for i in range(3):
            print(" | ".join(b[i]))
            if i<2: print("--+---+--")
        print("\n")
    def check_win(b, symbol):
        for i in range(3):
            if all(b[i,:]==symbol) or all(b[:,i]==symbol): return True
        if all(np.diag(b)==symbol) or all(np.diag(np.fliplr(b))==symbol): return True
        return False
    def available_moves(b):
        moves=[]
        for i in range(3):
            for j in range(3):
                if b[i,j]==' ': moves.append((i,j))
        return moves
    def computer_move(b, comp_symbol, user_symbol):
        moves = available_moves(b)
        for (i,j) in moves:
            temp = b.copy(); temp[i,j]=comp_symbol
            if check_win(temp, comp_symbol): return (i,j)
        for (i,j) in moves:
            temp = b.copy(); temp[i,j]=user_symbol
            if check_win(temp, user_symbol): return (i,j)
        for i in range(3):
            row = b[i]
            if list(row).count(comp_symbol)==2 and ' ' in row: return (i, list(row).index(' '))
        for j in range(3):
            col = b[:,j]
            if list(col).count(comp_symbol)==2 and ' ' in col: return (list(col).index(' '), j)
        diag1 = [b[i,i] for i in range(3)]
        if diag1.count(comp_symbol)==2 and ' ' in diag1: idx = diag1.index(' '); return (idx, idx)
        diag2 = [b[i,2-i] for i in range(3)]
        if diag2.count(comp_symbol)==2 and ' ' in diag2: idx = diag2.index(' '); return (idx, 2-idx)
        return random.choice(moves) if moves else None

    scoreboard = {"User":0, "Computer":0}
    print("Welcome to Tic Tac Toe!\n")
    user_name = input("Enter your name: ").strip()
    user_symbol = input("Choose your symbol (X or O): ").upper()
    while user_symbol not in ['X','O']:
        user_symbol = input("Invalid! Choose X or O: ").upper()
    comp_symbol = 'O' if user_symbol=='X' else 'X'
    print(f"\n{user_name} is '{user_symbol}' | Computer is '{comp_symbol}'")
    board = create_board(); print_board(board)
    turn = random.choice(['User','Computer']); print(f"{turn} will start!\n")
    for _ in range(9):
        if turn=='User':
            try:
                row = int(input("Enter row (0-2): ")); col = int(input("Enter column (0-2): "))
            except ValueError:
                print("Invalid input! Try again."); continue
            if 0<=row<3 and 0<=col<3 and board[row,col]==' ':
                board[row,col]=user_symbol
            else:
                print("Invalid move! Cell taken or out of range."); continue
            print_board(board)
            if check_win(board, user_symbol):
                print(f"🎉 {user_name} won!"); scoreboard["User"]+=1; break
            turn='Computer'
        else:
            print("Computer's turn..."); move = computer_move(board, comp_symbol, user_symbol)
            if move:
                board[move]=comp_symbol; print_board(board)
                if check_win(board, comp_symbol):
                    print("💻 Computer won!"); scoreboard["Computer"]+=1; break
            turn='User'
    else:
        print("It's a tie!")
    print(f"\n📊 Scoreboard:\n{user_name}: {scoreboard['User']} | Computer: {scoreboard['Computer']}")
    input("\nPress Enter to return to the Arcade Menu...")


In [8]:

# --- Arcade Menu ---
def arcade_menu():
    games = {
        "1": ("Hand Cricket", hand_cricket_main),
        "2": ("Hangman", hangman_main),
        "3": ("Battleship", battleship_main),
        "4": ("Blackjack", blackjack_main),
        "5": ("Typing Speed", typing_speed_main),
        "6": ("Rock Paper Scissors", rps_main),
        "7": ("Tic Tac Toe", tictactoe_main)
    }

    while True:
        print("\n===== Welcome to Python Arcade (Notebook) =====\n")
        for k in sorted(games.keys(), key=int):
            print(f"{k}. {games[k][0]}")
        print("0. Exit Arcade")

        choice = input("\nEnter the number of the game you want to play: ").strip()
        if choice == "0":
            print("Thanks for playing! Goodbye!")
            break
        if choice in games:
            try:
                print(f"Launching {games[choice][0]}...\n")
                games[choice][1]()  # call function
            except Exception as e:
                print("An error occurred while running the game:", e)
        else:
            print("Invalid choice! Enter a number from 0 to 7.")

# Start the menu
arcade_menu()



===== Welcome to Python Arcade (Notebook) =====

1. Hand Cricket
2. Hangman
3. Battleship
4. Blackjack
5. Typing Speed
6. Rock Paper Scissors
7. Tic Tac Toe
0. Exit Arcade

Enter the number of the game you want to play: 0
Thanks for playing! Goodbye!
